In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set seed for reproducibility
np.random.seed(42)

# Portfolio size
n_samples = 50000

# 1. Generate Application Data
loan_ids = [f"LN{str(i).zfill(6)}" for i in range(1, n_samples + 1)]
loan_products = np.random.choice(['Personal Loan', 'Two-Wheeler', 'Consumer Durable'], n_samples, p=[0.5, 0.3, 0.2])

# Simulate realistic Indian monthly incomes (Log-normal distribution for right-skewed salaries)
monthly_income_inr = np.random.lognormal(mean=10.8, sigma=0.6, size=n_samples) # Roughly peaks around 45k INR
monthly_income_inr = np.clip(monthly_income_inr, 15000, 500000).round(-2)

# Simulate CIBIL Scores (Skewed towards 700-750 for typical sourced portfolios)
cibil_scores = np.random.normal(loc=720, scale=80, size=n_samples)
cibil_scores = np.clip(cibil_scores, 300, 900).astype(int)

# Simulate Loan Amounts based on product
loan_amount_inr = np.where(
    loan_products == 'Personal Loan', np.random.uniform(50000, 1000000, n_samples),
    np.where(loan_products == 'Two-Wheeler', np.random.uniform(40000, 150000, n_samples),
             np.random.uniform(15000, 80000, n_samples))
).round(-3)

# Interest Rates based on CIBIL and Product
interest_rates = 24.0 - ((cibil_scores - 300) / 600) * 10 
interest_rates = np.clip(interest_rates, 10.5, 26.0).round(2)

# Calculate existing obligations to get DTI (Debt-to-Income)
existing_emi = monthly_income_inr * np.random.uniform(0.1, 0.5, n_samples)
new_emi = (loan_amount_inr * (interest_rates/1200)) / (1 - (1 + interest_rates/1200)**(-36)) # Assuming 36m tenure
dti_ratio = ((existing_emi + new_emi) / monthly_income_inr).round(3)

# 2. Simulate Performance & Delinquency Data
# Base probability of default increases with low CIBIL and high DTI
base_risk = ((900 - cibil_scores) / 600) * 0.6 + (dti_ratio * 0.4)
dpd_probabilities = np.random.rand(n_samples)

days_past_due = np.zeros(n_samples, dtype=int)

# Assign DPD based on risk threshold
for i in range(n_samples):
    if dpd_probabilities[i] < base_risk[i] * 0.1:
        days_past_due[i] = np.random.randint(90, 180) # Stage 3 (NPA/Default)
    elif dpd_probabilities[i] < base_risk[i] * 0.25:
        days_past_due[i] = np.random.randint(31, 89)  # Stage 2
    elif dpd_probabilities[i] < base_risk[i] * 0.5:
        days_past_due[i] = np.random.randint(1, 30)   # Stage 1 (Early Delinquency)

# Assign origination dates for Vintage Analysis (Last 24 months)
end_date = datetime(2026, 8, 1)
start_date = end_date - timedelta(days=730)
origination_dates = [start_date + timedelta(days=np.random.randint(0, 730)) for _ in range(n_samples)]

# Compile Dataset
df = pd.DataFrame({
    'loan_id': loan_ids,
    'origination_date': origination_dates,
    'loan_product': loan_products,
    'monthly_income_inr': monthly_income_inr,
    'cibil_score': cibil_scores,
    'loan_amount_inr': loan_amount_inr,
    'interest_rate_pct': interest_rates,
    'dti_ratio': dti_ratio,
    'days_past_due': days_past_due
})

# IND-AS 109 Stage Mapping
def assign_stage(dpd):
    if dpd == 0: return 'Current'
    elif dpd <= 30: return 'Stage 1'
    elif dpd <= 90: return 'Stage 2'
    else: return 'Stage 3 (Default)'

df['ind_as_stage'] = df['days_past_due'].apply(assign_stage)
df['default_flag'] = np.where(df['days_past_due'] > 90, 1, 0)

# Export
df.to_csv('synthetic_indian_loan_portfolio.csv', index=False)
print("Dataset successfully generated and saved.")

Dataset successfully generated and saved.


In [2]:
#Applying the Credit Policy (Approval Analytics)

import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('synthetic_indian_loan_portfolio.csv')

# Define standard Indian retail banking credit policy cut-offs
CIBIL_CUTOFF = 650
DTI_CUTOFF = 0.55  # 55%
MIN_INCOME_PL = 25000  # Minimum 25k INR for Personal Loans

def apply_credit_policy(row):
    """Returns True if approved, False if rejected based on policy."""
    if row['cibil_score'] < CIBIL_CUTOFF:
        return False, "Reject: Low CIBIL"
    if row['dti_ratio'] > DTI_CUTOFF:
        return False, "Reject: High DTI"
    if row['loan_product'] == 'Personal Loan' and row['monthly_income_inr'] < MIN_INCOME_PL:
        return False, "Reject: Income Policy (PL)"
    return True, "Approved"

# Apply policy
df[['is_approved', 'decision_reason']] = df.apply(
    lambda row: pd.Series(apply_credit_policy(row)), axis=1
)

# Calculate Approval Metrics
total_apps = len(df)
approved_apps = df['is_approved'].sum()
approval_rate = (approved_apps / total_apps) * 100

print(f"Total Applications: {total_apps:,}")
print(f"Total Approved: {approved_apps:,}")
print(f"Overall Approval Rate: {approval_rate:.2f}%\n")

# Rejection Breakdown
print("Rejection Reasons:")
print(df[df['is_approved'] == False]['decision_reason'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

Total Applications: 50,000
Total Approved: 26,027
Overall Approval Rate: 52.05%

Rejection Reasons:
decision_reason
Reject: High DTI              59.35%
Reject: Low CIBIL              39.6%
Reject: Income Policy (PL)     1.06%
Name: proportion, dtype: str


In [16]:
# Risk Segmentation (Sourced Portfolio)

# Filter for only approved/booked loans
booked_portfolio = df[df['is_approved'] == True].copy()

# Define Risk Tiers based on standard bureau brackets
def assign_risk_tier(cibil):
    if cibil >= 800: return 'Tier 1: Prime Plus'
    elif cibil >= 750: return 'Tier 2: Prime'
    elif cibil >= 700: return 'Tier 3: Near Prime'
    else: return 'Tier 4: Sub-Prime'

booked_portfolio['risk_tier'] = booked_portfolio['cibil_score'].apply(assign_risk_tier)

# Calculate Portfolio Quality by Risk Tier
segmentation_summary = booked_portfolio.groupby('risk_tier').agg(
    total_loans=('loan_id', 'count'),
    avg_loan_amount=('loan_amount_inr', 'mean'),
    avg_dti=('dti_ratio', 'mean'),
    npa_count=('default_flag', 'sum') # 90+ DPD
).reset_index()

# Calculate Default Rate per Tier
segmentation_summary['default_rate_pct'] = (segmentation_summary['npa_count'] / segmentation_summary['total_loans'] * 100).round(2)

# Display the summary table
print("\nPortfolio Risk Segmentation:")
print(segmentation_summary)

import os
output_dir = 'data/processed'

# 2. Create the directory (and any parent directories) if they don't exist
os.makedirs(output_dir, exist_ok=True)

# 3. Save the cleaned and approved portfolio
booked_portfolio.to_csv(f'{output_dir}/booked_portfolio.csv', index=False)

print(f"Success! File saved to {output_dir}/booked_portfolio.csv")


Portfolio Risk Segmentation:
            risk_tier  total_loans  avg_loan_amount   avg_dti  npa_count  \
0  Tier 1: Prime Plus         5112    149863.458529  0.362337        115   
1       Tier 2: Prime         6406    144044.333437  0.363996        172   
2  Tier 3: Near Prime         7892    146656.614293  0.366314        266   
3   Tier 4: Sub-Prime         6617    141350.309808  0.362659        256   

   default_rate_pct  
0              2.25  
1              2.68  
2              3.37  
3              3.87  
Success! File saved to data/processed/booked_portfolio.csv
